# 01 — Reanálise reproduzível (CPU)
Use **Ambiente de execução → Executar tudo**. Autorize o Drive quando solicitado. Instalação, revisões, dados, treino, avaliação e retomada são automáticos. O perfil **quick** é um teste curto de infraestrutura; não é o estudo completo. **pilot** e **full** aumentam o trabalho e ficam em pastas distintas. Não é necessário alterar flags de execução nem fornecer chave de API.

Este notebook não treina modelos.


In [ ]:
# Execute esta célula. As opções abaixo já estão prontas para o teste curto.
PROFILE = "quick" # @param ["quick", "pilot", "full"]
SALVAR_NO_DRIVE = True # @param {type:"boolean"}

from pathlib import Path
import json, os, subprocess, sys, tempfile, shutil
SOURCE_REF = "d3abb149ced4c663c01e621644f2bedca7313e05"
NEEDS_GPU = False
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

def bootstrap_run(command):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout, flush=True)
    if result.returncode:
        raise RuntimeError("Falha ao preparar o código do laboratório:\n" + result.stdout + result.stderr)

override = os.environ.get("COT_PROJECT_ROOT")
if override:
    ROOT = Path(override).resolve()
elif ON_COLAB:
    cache = Path("/content/cot-lab/source")
    cache.mkdir(parents=True, exist_ok=True)
    checkout = cache / SOURCE_REF
    if not (checkout / ".git").is_dir():
        temporary = Path(tempfile.mkdtemp(prefix="download-", dir=cache))
        try:
            bootstrap_run(["git", "init", "--quiet", str(temporary)])
            bootstrap_run(["git", "-C", str(temporary), "fetch", "--quiet", "--depth", "1",
                "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git", SOURCE_REF])
            bootstrap_run(["git", "-C", str(temporary), "checkout", "--quiet", "--detach", "FETCH_HEAD"])
            temporary.rename(checkout)
        finally:
            if temporary.exists():
                shutil.rmtree(temporary)
    actual_ref = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    assert actual_ref == SOURCE_REF, "Checkout diverge do notebook. Abra a versão atual no GitHub."
    ROOT = checkout / "revisao_2026"
else:
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd / "revisao_2026", cwd.parent / "revisao_2026"]
    ROOT = next((p for p in candidates if (p / "src/colab_runtime.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Fora do Colab, abra o notebook dentro do clone do repositório.")

assert (ROOT / "src/colab_runtime.py").is_file(), "Código do laboratório não encontrado."
if os.environ.get("COT_STORAGE_ROOT"):
    STORAGE = Path(os.environ["COT_STORAGE_ROOT"])
elif ON_COLAB and SALVAR_NO_DRIVE:
    drive.mount("/content/drive")
    STORAGE = Path("/content/drive/MyDrive/CoT_Obfuscation_Lab/colab_v3")
else:
    STORAGE = Path("/content/cot-lab/results") if ON_COLAB else Path.home()/"CoT_Obfuscation_Lab/colab_v3"
    if ON_COLAB:
        print("Resultados nesta sessão: baixe o ZIP final antes de encerrar o Colab.")
STORAGE.mkdir(parents=True, exist_ok=True)
os.environ["COT_SOURCE_REF"] = SOURCE_REF
sys.path.insert(0, str(ROOT / "src"))
from colab_runtime import ColabSession
SESSION = ColabSession(ROOT, STORAGE, PROFILE, gpu=NEEDS_GPU)
(STORAGE / ("session_"+PROFILE+".json")).write_text(json.dumps({"source_ref":SOURCE_REF,
    "profile":PROFILE,"persistent_drive":bool(ON_COLAB and SALVAR_NO_DRIVE),"storage":str(STORAGE)}, indent=2))
print("Código:", SOURCE_REF[:12], "| Perfil:", PROFILE)
print("Resultados:", STORAGE)


In [ ]:
SESSION.ensure_environment()


In [ ]:
SESSION.command(["-m","unittest","discover","-s","tests","-v"], "01_tests.log")
SESSION.command(["src/reanalyse.py","--output-root",str(STORAGE/"reanalysis")], "01_reanalysis.log")


In [ ]:
for name in ["summary.csv","contrasts.csv","grading_stress_summary.csv"]:
    SESSION.show_csv(STORAGE/"reanalysis/results"/name)
from IPython.display import display, Image
for path in sorted((STORAGE/"reanalysis/figures").glob("*.png")):
    display(Image(data=path.read_bytes()))


In [ ]:
SESSION.bundle("reanalysis")


Os dados históricos contêm métricas, sem as 540 gerações integrais. Os testes do corretor são mecânicos.
